# Practical 06 — Fourier Neural Operators: build, evaluate, diagnose

**Student notebook — 90-minute practical.** Complete the TODOs, answer the questions, and run the cells in order. The spectral-convolution bonus at the end is optional.

**Learning objectives:** (1) contrast PCA-Net and FNO representations; (2) **build an FNO from its layers**; (3) check tensor shapes and cross-resolution predictions; (4) diagnose missing boundary information and recursive rollout error.

| Section | Time |
|---|---:|
| PCA-Net → FNO: physical problem and representation | 10 min |
| **Construct the FNO** + verify shapes | **25 min** |
| Train and test across resolutions | 15 min |
| Debug A — missing boundary conditions | 20 min |
| Debug B — recursive rollout | 15 min |
| Final synthesis | 5 min |
| **Total** | **90 min** |


**Workflow:** predict → implement → check → evaluate → diagnose. The numerical solvers are provided so the emphasis stays on operator learning. Training time depends on hardware; `QUICK_RUN=True` only checks execution and **is not suitable for interpreting accuracy**.

Reference: [Li et al. (2021), Fourier Neural Operator for Parametric Partial Differential Equations](https://arxiv.org/abs/2010.08895).

In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 123
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# Small 1D FFTs can be slower when many CPU threads are used.
if DEVICE == 'cpu':
    torch.set_num_threads(min(2, torch.get_num_threads()))

# QUICK_RUN=True checks that the code executes but is not a meaningful training run.
QUICK_RUN = False
EPOCHS_PART1 = 20 if QUICK_RUN else 300
EPOCHS_DEBUG = 20 if QUICK_RUN else 300
PATIENCE = 15 if QUICK_RUN else 70

print('Device:', DEVICE, '| quick run:', QUICK_RUN)

In [ ]:
def relative_l2(pred, ref):
    '''Relative discrete L2 error for each sample (last dimension is space).'''
    pred = torch.as_tensor(pred)
    ref = torch.as_tensor(ref)
    return (torch.linalg.vector_norm(pred-ref, dim=-1) /
            torch.linalg.vector_norm(ref, dim=-1).clamp_min(1e-12))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def make_loaders(X, Y, n_train=500, n_val=100, batch_size=64):
    '''First train, then validation, then held-out test. No test-set tuning.'''
    X = torch.as_tensor(X, dtype=torch.float32)
    Y = torch.as_tensor(Y, dtype=torch.float32)
    assert len(X) == len(Y) and n_train+n_val < len(X)
    tr = DataLoader(TensorDataset(X[:n_train], Y[:n_train]),
                    batch_size=batch_size, shuffle=True)
    va = DataLoader(TensorDataset(X[n_train:n_train+n_val],
                                  Y[n_train:n_train+n_val]),
                    batch_size=batch_size, shuffle=False)
    return tr, va, X[n_train+n_val:], Y[n_train+n_val:]


def train_model(model, train_loader, val_loader, epochs, lr=3e-3, patience=110):
    '''Adam + early stopping on *validation* MSE; restore best parameters.'''
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'train': [], 'val': []}
    best_val, best_state, best_epoch = float('inf'), None, 0

    for epoch in range(epochs):
        model.train()
        total, seen = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            loss = F.mse_loss(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item()*len(xb)
            seen += len(xb)
        history['train'].append(total/seen)

        model.eval()
        total, seen = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                loss = F.mse_loss(model(xb), yb)
                total += loss.item()*len(xb)
                seen += len(xb)
        val = total/seen
        history['val'].append(val)
        if val < best_val:
            best_val, best_epoch = val, epoch
            best_state = copy.deepcopy(model.state_dict())
        elif epoch-best_epoch >= patience:
            break

    model.load_state_dict(best_state)
    return model, history


def predict(model, X):
    model.eval()
    X = torch.as_tensor(X, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        y = model(X)
    return y.cpu()


def fit_case(X, Y, *, modes=10, width=4, epochs=None, depth=1):
    '''Same train/validation/test split for controlled input comparisons.'''
    epochs = EPOCHS_DEBUG if epochs is None else epochs
    tr, va, X_test, Y_test = make_loaders(X, Y, N_TRAIN, N_VAL)
    torch.manual_seed(SEED)
    model = FNO1d(in_features=X.shape[1], width=width,
                  modes=modes, depth=depth)
    model, history = train_model(model, tr, va, epochs=epochs,
                                 patience=PATIENCE)
    prediction = predict(model, X_test)
    return model, history, prediction, Y_test

# Part 1 — Build and evaluate an FNO (50 min)

## 1. From PCA-Net to an FNO (10 min)

In **Practical 05**, PCA-Net represented input and output fields with **global PCA coordinates**:

$$u_0(x)\xrightarrow{E_{\mathrm{in}}}z_0\in\mathbb R^{r_{\rm in}}
\xrightarrow{\mathrm{MLP}_\theta}\hat z_T\in\mathbb R^{r_{\rm out}}
\xrightarrow{D_{\rm out}}\hat u_T(x).$$

Here we keep a **spatially distributed latent field**:

$$u(x_i)\xrightarrow{P}v_0(x_i)\in\mathbb R^{d_v}
\xrightarrow{\text{FNO blocks}}v_L(x_i)\xrightarrow{Q}\hat u(x_i).$$

The PDE is **viscous Burgers** in both practicals, but **the learned maps differ**:

| | Practical 05 — PCA-Net | Practical 06 — FNO |
|---|---|---|
| Time mapping | $u(0)\mapsto u(0.4)$ | $u(0)\mapsto u(0.1)$; later used recursively |
| Representation | PCA coefficients $z\in\mathbb R^r$ | latent field $v\in\mathbb R^{d_v\times N}$ |
| Training resolution | $N=128$ | $N=32$; tested at $64$ and $128$ |
| Main inductive bias | low-dimensional **linear subspace** | nonlocal interactions parametrized in Fourier space |

Note: training on pairs from $t=0$ does **not** mean the FNO is trained on the complete family of states encountered in a long rollout.

### Question 1 — check the representation

1. Which model discards the spatial grid **inside its learned latent dynamics**?
2. In which model is the first transformation a **global dimensionality reduction**? In which is it a **pointwise lifting**?
3. If the spatial resolution changes, which internal representation naturally changes its number of spatial samples?

### Numerical data — Burgers equation (solver supplied)

We solve periodic viscous Burgers on $x\in[0,2\pi)$:

$$\partial_t u+u\partial_xu=\nu\partial_{xx}u,\qquad \nu=0.08.$$

- **Initial conditions:** random smooth Fourier series with wavenumbers $1,\ldots,5$ (the **same sampled functions** are evaluated on each grid).
- **Space:** Fourier pseudospectral differentiation; calculate the nonlinear flux $u^2/2$ in physical space and filter its Fourier coefficients at high frequencies ($2/3$ cutoff).
- **Time:** explicit fourth-order Runge–Kutta (RK4), with internal steps $\leq0.002$, to $\Delta t=0.1$.
- **Data:** 500 training / 100 validation / 100 test pairs. The reference solver is rerun at 32 and 64 points, and on the 128-point **test** set (not interpolated from a coarse FNO prediction).

These numerical solves produce the **targets**; the trained FNO will make predictions without executing the PDE solver. This is the same PDE family as Practical 05 but a **shorter time horizon and different training resolution**.

In [ ]:
LENGTH = 2.0 * np.pi
VISCOSITY = 0.08
OPERATOR_DT = 0.1

N_LOW = 32
N_HIGH = 64
N_FINE = 128  # evaluation only

N_TRAIN = 500
N_VAL = 100
N_TEST = 100
N_TOTAL = N_TRAIN + N_VAL + N_TEST


def sample_burgers_parameters(
    n_samples,
    max_freq=5,
    seed=10,
):
    rng = np.random.default_rng(seed)

    k = np.arange(
        1,
        max_freq + 1,
        dtype=np.float64,
    )

    scale = 1.0 / k**2

    a = (
        rng.normal(
            size=(n_samples, max_freq)
        )
        * scale[None, :]
    )

    b = (
        rng.normal(
            size=(n_samples, max_freq)
        )
        * scale[None, :]
    )

    amplitude = rng.uniform(
        0.75,
        1.10,
        size=(n_samples, 1),
    )

    return a, b, amplitude


def evaluate_burgers_initial_conditions(
    parameters,
    nx,
):
    a, b, amplitude = parameters

    x = np.linspace(
        0.0,
        LENGTH,
        nx,
        endpoint=False,
    )

    k = np.arange(
        1,
        a.shape[1] + 1,
    )

    phase = (
        x[:, None]
        * k[None, :]
    )

    u0 = (
        a @ np.sin(phase).T
        + b @ np.cos(phase).T
    )

    u0 *= amplitude

    return (
        x.astype(np.float32),
        u0.astype(np.float32),
    )


def solve_viscous_burgers(
    u0,
    nu=VISCOSITY,
    final_time=OPERATOR_DT,
    dt_max=2e-3,
):
    u = np.asarray(
        u0,
        dtype=np.float64,
    ).copy()

    nx = u.shape[-1]

    wave_number = np.fft.fftfreq(
        nx,
        d=1.0 / nx,
    )

    cutoff = nx // 3

    n_steps = int(
        np.ceil(
            final_time / dt_max
        )
    )

    dt = final_time / n_steps

    def rhs(v):

        v_hat = np.fft.fft(
            v,
            axis=-1,
        )

        flux_hat = np.fft.fft(
            0.5 * v**2,
            axis=-1,
        )

        flux_hat[
            ...,
            np.abs(wave_number) > cutoff
        ] = 0.0

        return np.fft.ifft(
            -1j
            * wave_number
            * flux_hat
            - nu
            * wave_number**2
            * v_hat,
            axis=-1,
        ).real

    for _ in range(n_steps):

        k1 = rhs(u)
        k2 = rhs(u + 0.5*dt*k1)
        k3 = rhs(u + 0.5*dt*k2)
        k4 = rhs(u + dt*k3)

        u += (
            dt
            * (
                k1
                + 2*k2
                + 2*k3
                + k4
            )
            / 6.0
        )

    return u.astype(np.float32)


parameters = sample_burgers_parameters(
    N_TOTAL
)

x32, u0_32 = evaluate_burgers_initial_conditions(
    parameters,
    N_LOW,
)

x64, u0_64 = evaluate_burgers_initial_conditions(
    parameters,
    N_HIGH,
)

u1_32 = solve_viscous_burgers(
    u0_32
)

u1_64 = solve_viscous_burgers(
    u0_64
)

# The SAME held-out initial-condition parameters, sampled at a finer grid.
test_parameters = tuple(p[N_TRAIN + N_VAL:] for p in parameters)
x128, u0_128_test = evaluate_burgers_initial_conditions(test_parameters, N_FINE)
u1_128_test = solve_viscous_burgers(u0_128_test)

# Same physical initial-condition family across grids.
assert u0_32.shape == (N_TOTAL, 32) and u0_64.shape == (N_TOTAL, 64)
assert u1_32.shape == u0_32.shape and u1_64.shape == u0_64.shape
assert u1_128_test.shape == (N_TEST, 128)


In [ ]:
fig, axes = plt.subplots(
    2,
    4,
    figsize=(13, 5.6),
    sharey=True,
    constrained_layout=True,
)

for col, j in enumerate([0, 1, 2, 3]):

    axes[0, col].plot(
        x32,
        u0_32[j],
        label="$u(t)$",
    )

    axes[0, col].plot(
        x32,
        u1_32[j],
        "--",
        label="$u(t+\\Delta t)$",
    )

    axes[0, col].set_title(
        f"$N=32$, sample {j}"
    )

    axes[1, col].plot(
        x64,
        u0_64[j],
    )

    axes[1, col].plot(
        x64,
        u1_64[j],
        "--",
    )

    axes[1, col].set_title(
        f"$N=64$, same sample {j}"
    )

    axes[0, col].grid(alpha=0.2)
    axes[1, col].grid(alpha=0.2)

for ax in axes[1]:
    ax.set_xlabel("$x$")

axes[0, 0].set_ylabel("$u(x)$")
axes[1, 0].set_ylabel("$u(x)$")

handles, labels = axes[0, 0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="outside upper center",
    ncol=2,
    frameon=False,
)

plt.show()

## 2. Construct an FNO (25 min)

A generic neural operator layer updates the spatial feature field using local and nonlocal interactions:

$$v_{\ell+1}(x)=\sigma\left[Wv_\ell(x)+\int_\Omega\kappa_\theta(x,y)v_\ell(y)\,dy\right].$$

For **homogeneous periodic** problems, we can use the translation-invariant kernel $\kappa_\theta(x,y)=\kappa_\theta(x-y)$ and the convolution theorem:

$$\boxed{v_{\ell+1}=\sigma\left(Wv_\ell+\mathcal F^{-1}\!\left[R_\theta(k)\,\mathcal F[v_\ell](k)\right]\right).}$$

We will **provide the spectral convolution** and build the rest of the model yourselves. Use a channel-first layout $(B,C,N)$, where $B$ is batch size, $C$ channels, and $N$ spatial points.

- **Lift $P$**: a pointwise $1\times1$ convolution, from `in_features` channels to `width` channels.
- **Global branch**: `SpectralConv1d(width, width, modes)`.
- **Local branch $W$**: a pointwise $1\times1$ convolution from `width` to `width` channels.
- **Activation $\sigma$**: GELU, after the **sum** of both branches.
- **Project $Q$**: a pointwise $1\times1$ convolution from `width` channels back to 1 output channel.

The lift, local mixing and projection act at **every spatial point**, whereas the Fourier branch couples spatial locations nonlocally. The spectral branch retains only `modes` positive rFFT coefficients; the **complete FNO output is not necessarily band-limited**.

### Provided code — spectral convolution

Read the layer below before implementing `FNO1d`. The learnable complex weight has shape `(input_channels, output_channels, modes)` and mixes channels separately at each retained Fourier frequency.

The inverse FFT returns a real-valued field with the **same number of spatial points**.

In [ ]:
class SpectralConv1d(nn.Module):
    '''1D Fourier convolution: learn channel mixing for low positive rFFT modes.'''
    def __init__(self, in_channels, out_channels, modes):
        super().__init__()
        self.modes = modes
        scale = 1.0/(in_channels*out_channels)
        self.weight = nn.Parameter(scale*torch.randn(in_channels, out_channels,
                                                     modes, dtype=torch.cfloat))

    def forward(self, x):                       # (B, Cin, N)
        N = x.shape[-1]
        x_ft = torch.fft.rfft(x, dim=-1)         # (B, Cin, N//2+1)
        out_ft = torch.zeros(x.shape[0], self.weight.shape[1], x_ft.shape[-1],
                             device=x.device, dtype=x_ft.dtype)
        m = min(self.modes, x_ft.shape[-1])
        out_ft[:, :, :m] = torch.einsum('bim,iom->bom',
                                       x_ft[:, :, :m], self.weight[:, :, :m])
        return torch.fft.irfft(out_ft, n=N, dim=-1)

### Coding exercises 1 and 2 — build `FNO1d`

**Exercise 1 — define the modules (≈10 min).** In `__init__`, create:

1. `self.lift`: pointwise lifting $P$;
2. `self.spectral_layers`: a `nn.ModuleList` with `depth` spectral layers;
3. `self.local_layers`: a `nn.ModuleList` with `depth` pointwise channel-mixing layers;
4. `self.project`: pointwise projection $Q$. The GELU activation is supplied.

**Exercise 2 — implement `forward` (≈10 min).** For an input of shape `(B, in_features, N)`:

1. Lift it to `(B, width, N)`.
2. Repeat `depth` times: `v = activation(spectral(v) + local(v))`.
3. Project the latent field back to **one** channel and return shape `(B, N)`.

**Hints:** `nn.Conv1d(..., kernel_size=1)` is pointwise; `nn.ModuleList([... for _ in range(depth)])` creates a trainable stack. You should **not** use a `Linear(N, ...)` layer: it would tie the parameters to a particular resolution.

In [ ]:
class FNO1d(nn.Module):
    def __init__(self, in_features=1, width=16, modes=10, depth=2):
        super().__init__()
        # TODO 1: define self.lift (pointwise Conv1d: in_features -> width)
        # TODO 2: define self.spectral_layers (ModuleList of depth SpectralConv1d)
        # TODO 3: define self.local_layers (ModuleList of depth pointwise Conv1d)
        self.activation = nn.GELU()
        # TODO 4: define self.project (pointwise Conv1d: width -> 1)
        raise NotImplementedError("Complete the FNO modules before running.")

    def forward(self, x):
        # TODO 5: lift x to a latent feature field v
        # TODO 6: for each block, add spectral(v) and local(v), then apply activation
        # TODO 7: project to 1 channel and return a tensor with shape (B, N)
        raise NotImplementedError("Complete the FNO forward pass before running.")

### Quick implementation checks

Run the checks before training. They should succeed at **two spatial resolutions** and with **two FNO blocks**, without changing the class or its parameters.

In [ ]:
torch.manual_seed(SEED)
check_model = FNO1d(in_features=1, width=8, modes=6, depth=2).to(DEVICE)
assert len(check_model.spectral_layers) == len(check_model.local_layers) == 2
with torch.no_grad():
    for n_points in (32, 64):
        y_check = check_model(torch.randn(3, 1, n_points, device=DEVICE))
        assert y_check.shape == (3, n_points), y_check.shape
        assert torch.isfinite(y_check).all()
print("FNO architecture checks passed (N=32 and 64, depth=2).")

### Question 2 — local versus nonlocal interactions

1. Which operations mix **only the feature channels at one spatial point**?
2. Which operation introduces **nonlocal interactions** between spatial points?
3. Are the trainable weights of this architecture tied to a particular number of spatial points $N$?

### Exercise 3 — tensor shapes (≈5 min)

First **predict** the shapes for batch size $B=4$, width $d_v=16$ and 10 retained Fourier modes. Then run the cell below to verify.

| Tensor or weight | $N=32$ | $N=64$ |
|---|---|---|
| Input | ? | ? |
| Lifted latent field | ? | ? |
| rFFT of latent field | ? | ? |
| `lift.weight` | ? | ? |
| `local_layers[0].weight` | ? | ? |
| `spectral_layers[0].weight` | ? | ? |
| `project.weight` | ? | ? |

**Two precise questions:** (i) Does `rfft` add a tensor axis? (ii) Why is the frequency axis of an even-length real signal of size $N/2+1$?

The shape demonstration uses `width=16`; the training experiments use `width=4` and `depth=1` to reduce cost. The implementation check above deliberately uses `depth=2` to test the block loop.

In [ ]:
torch.manual_seed(SEED)

shape_model = FNO1d(
    in_features=1,
    width=16,
    modes=10,
    depth=1,
).to(DEVICE)

for N_here, u_here in [
    (32, u0_32),
    (64, u0_64),
]:

    x_batch = torch.tensor(
        u_here[:4, None, :],
        dtype=torch.float32,
        device=DEVICE,
    )

    with torch.no_grad():

        v = shape_model.lift(
            x_batch
        )

        v_hat = torch.fft.rfft(
            v,
            dim=-1,
        )

        y = shape_model(
            x_batch
        )

    print(
        f"N={N_here}",
        "| input:", tuple(x_batch.shape),
        "| latent:", tuple(v.shape),
        "| FFT:", tuple(v_hat.shape),
        "| output:", tuple(y.shape),
    )

print("lift weights :", tuple(shape_model.lift.weight.shape))
print("local W      :", tuple(shape_model.local_layers[0].weight.shape))
print("spectral     :", tuple(shape_model.spectral_layers[0].weight.shape))
print("projection   :", tuple(shape_model.project.weight.shape))

## 3. Train once, evaluate at multiple resolutions (15 min)

Train an FNO only at **$N=32$**. Then evaluate the **same trained weights** on the 32-, 64- and 128-point **test fields**, with reference solutions recomputed on each grid.

Compare against the no-learning **identity baseline** $\widehat u(t+\Delta t)=u(t)$.

**Prediction before running:** Does changing the grid length require changing the number or shape of learnable parameters? Does a similar error at several grids prove convergence toward the continuous operator?

*With `QUICK_RUN=True`, the FNO may not outperform identity: this mode tests only that the implementation executes.*

In [ ]:
X32 = u0_32[:, None, :]
Y32 = u1_32
train_loader_32, val_loader_32, X32_test, Y32_test = make_loaders(
    X32, Y32, n_train=N_TRAIN, n_val=N_VAL
)

torch.manual_seed(SEED)
fno_burgers = FNO1d(in_features=1, width=4, modes=10, depth=1)
fno_burgers, history_burgers = train_model(
    fno_burgers, train_loader_32, val_loader_32,
    epochs=EPOCHS_PART1, patience=PATIENCE
)
pred32 = predict(fno_burgers, X32_test)
X64_test = torch.tensor(u0_64[N_TRAIN+N_VAL:, None, :], dtype=torch.float32)
Y64_test = torch.tensor(u1_64[N_TRAIN+N_VAL:], dtype=torch.float32)
pred64 = predict(fno_burgers, X64_test)
X128_test = torch.as_tensor(u0_128_test[:, None, :], dtype=torch.float32)
Y128_test = torch.as_tensor(u1_128_test, dtype=torch.float32)
pred128 = predict(fno_burgers, X128_test)

err32 = relative_l2(pred32, Y32_test)
err64 = relative_l2(pred64, Y64_test)
err128 = relative_l2(pred128, Y128_test)
# No-learning baseline: predict u(t+dt) = u(t).
identity32 = relative_l2(X32_test[:, 0, :], Y32_test)
identity64 = relative_l2(X64_test[:, 0, :], Y64_test)
identity128 = relative_l2(X128_test[:, 0, :], Y128_test)

print(pd.DataFrame({
    'resolution': [32, 64, 128],
    'FNO relative L2': [err32.mean().item(), err64.mean().item(), err128.mean().item()],
    'identity relative L2': [identity32.mean().item(), identity64.mean().item(), identity128.mean().item()],
}).to_string(index=False, float_format=lambda z: f'{z:.4f}'))
print('Trainable parameters:', count_parameters(fno_burgers))

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(10.8, 3.7),
    constrained_layout=True,
)

axes[0].plot(
    history_burgers["train"],
    label="training",
)

axes[0].plot(
    history_burgers["val"],
    label="validation",
)

axes[0].set(
    xlabel="epoch",
    ylabel="MSE",
    title="Training at $N=32$",
)

axes[0].set_yscale("log")
axes[0].grid(alpha=0.2)
axes[0].legend(frameon=False)

axes[1].plot(
    [32, 64, 128],
    [float(err32.mean()), float(err64.mean()), float(err128.mean())],
    "o-", label="FNO",
)
axes[1].plot(
    [32, 64, 128],
    [float(identity32.mean()), float(identity64.mean()), float(identity128.mean())],
    "s--", label="identity baseline",
)
axes[1].legend(frameon=False)

axes[1].set(
    xticks=[32, 64, 128],
    xlabel="grid points",
    ylabel="mean relative $L^2$ error",
    title="Same learned weights",
)

axes[1].grid(alpha=0.2)

plt.show()

### Question 3 — interpret the results

1. Read the table: does the FNO beat the identity baseline at $N=32$, $64$ and $128$?
2. Why can the **same weights** be evaluated at those three resolutions?
3. Do similar errors across grids establish discretization convergence? What would you check before claiming it?

# Part 2 — Diagnose failures (35 min)

The two failures below have **different causes**. Before increasing the number of network parameters, check that the inputs specify the physical problem and that the model is evaluated under the same conditions as training.

## Debug A — Missing boundary-condition information (20 min)

Consider steady 1D Poisson:

$$-u''(x)=f(x),\qquad x\in[0,1],\qquad u(0)=b_0,\quad u(1)=b_1.$$

For example, $u$ is a stationary temperature field, $f$ a volumetric heat source and $b_0,b_1$ imposed endpoint temperatures.

We compare two equally structured FNOs with **different input information**:

- **Incomplete input:** $[f(x),x]$.
- **Complete input:** $[f(x),x,b_0,b_1]$, where the boundary values are repeated along the grid.

**Numerical reference:** second-order **centered finite differences** at 64 grid points; the discrete Poisson linear system is solved using NumPy. The random forcing $f$ is evaluated on the same grid $x=0,\ldots,1$.

**Important:** the Fourier kernel is periodic internally, but providing $x$ and BC channels allows the model to learn position-dependent outputs; it does **not** impose Dirichlet values exactly.

### Prediction A (before training)

Which FNO do you expect to predict the solution more accurately? Give **one reason**.

In [ ]:
def sample_forcing(n_samples, x, max_freq=8, rng=None):
    '''Smooth random Fourier forcing evaluated on the *given* physical coordinates.'''
    rng = np.random.default_rng(SEED) if rng is None else rng
    x = np.asarray(x, dtype=np.float64)
    f = np.zeros((n_samples, len(x)), dtype=np.float64)
    for k in range(1, max_freq+1):
        scale = 1.0/(1.0+k**1.5)
        a = rng.normal(size=(n_samples, 1))*scale
        b = rng.normal(size=(n_samples, 1))*scale
        f += a*np.cos(2*np.pi*k*x)[None, :]
        f += b*np.sin(2*np.pi*k*x)[None, :]
    return f.astype(np.float32)

In [ ]:
def solve_dirichlet_poisson_1d(
    f,
    bc0,
    bc1,
):
    f = np.asarray(
        f,
        dtype=np.float64,
    )

    bc0 = np.asarray(
        bc0,
        dtype=np.float64,
    )

    bc1 = np.asarray(
        bc1,
        dtype=np.float64,
    )

    _, N = f.shape

    h = 1.0 / (N - 1)
    ni = N - 2

    A = np.diag(
        2.0 / h**2
        * np.ones(ni)
    )

    off = (
        -1.0 / h**2
        * np.ones(ni - 1)
    )

    A += np.diag(off, 1)
    A += np.diag(off, -1)

    rhs = f[:, 1:-1].T.copy()

    rhs[0, :] += bc0 / h**2
    rhs[-1, :] += bc1 / h**2

    U = np.zeros_like(
        f,
        dtype=np.float32,
    )

    U[:, 0] = bc0
    U[:, -1] = bc1

    U[:, 1:-1] = np.linalg.solve(
        A,
        rhs,
    ).T

    return U


N_B = 64
x_B = np.linspace(
    0.0,
    1.0,
    N_B,
)

rng = np.random.default_rng(22)

fB_np = sample_forcing(
    N_TOTAL,
    x_B,
    max_freq=6,
    rng=rng,
)

b0 = rng.uniform(
    -0.6,
    0.6,
    size=N_TOTAL,
).astype(np.float32)

b1 = rng.uniform(
    -0.6,
    0.6,
    size=N_TOTAL,
).astype(np.float32)

uB_np = solve_dirichlet_poisson_1d(
    fB_np,
    b0,
    b1,
)

x_channel = np.repeat(
    x_B[None, :],
    N_TOTAL,
    axis=0,
).astype(np.float32)

b0_channel = np.repeat(
    b0[:, None],
    N_B,
    axis=1,
)

b1_channel = np.repeat(
    b1[:, None],
    N_B,
    axis=1,
)

**Sanity check.** For $f=0$, every affine function $u(x)=(1-x)b_0+xb_1$ solves the PDE. The same forcing can therefore generate different solutions.

In [ ]:
zero_force = np.zeros((2, N_B), dtype=np.float32)
zero_solutions = solve_dirichlet_poisson_1d(
    zero_force, np.array([0., 1.]), np.array([0., -1.])
)
print('Same f, different solutions?  ',
      not np.allclose(zero_solutions[0], zero_solutions[1]))

### Coding task — assemble the two inputs

Fill `XB_bad` and `XB_good` with channels ordered as below (axis 1 is the feature-channel dimension):

- `XB_bad`: $[f, x]$;
- `XB_good`: $[f, x, b_0, b_1]$.

The dimensions should be `(700,2,64)` and `(700,4,64)`.

In [ ]:
# TODO: stack the provided channels into (samples, channels, spatial points)
XB_bad = ...
XB_good = ...
assert XB_bad.shape == (N_TOTAL, 2, N_B)
assert XB_good.shape == (N_TOTAL, 4, N_B)

In [ ]:
fno_B_bad, h_B_bad, pred_B_bad, YB_te = fit_case(
    XB_bad, uB_np, modes=10, epochs=EPOCHS_DEBUG
)
fno_B_good, h_B_good, pred_B_good, YB2_te = fit_case(
    XB_good, uB_np, modes=10, epochs=EPOCHS_DEBUG
)
assert torch.equal(YB_te, YB2_te)
print('BC omitted    :', f'{relative_l2(pred_B_bad, YB_te).mean():.3%}')
print('BC supplied   :', f'{relative_l2(pred_B_good, YB_te).mean():.3%}')
print('Boundary MAE (supplied):',
      float((pred_B_good[:, [0, -1]]-YB_te[:, [0, -1]]).abs().mean()))

In [ ]:
indices = np.linspace(
    0,
    len(YB_te) - 1,
    8,
    dtype=int,
)

fig, axes = plt.subplots(
    2,
    4,
    figsize=(14, 6),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

axes = axes.ravel()

for ax, j in zip(
    axes,
    indices,
):

    ax.plot(
        x_B,
        YB_te[j],
        label="reference",
    )

    ax.plot(
        x_B,
        pred_B_bad[j],
        "--",
        label="BC omitted",
    )

    ax.plot(
        x_B,
        pred_B_good[j],
        ":",
        label="BC encoded",
    )

    ax.scatter(
        [0, 1],
        [
            YB_te[j, 0],
            YB_te[j, -1],
        ],
        zorder=5,
        label="boundary values",
    )

    ax.set_title(
        f"Test sample {j}"
    )

    ax.grid(alpha=0.2)

for ax in axes[-4:]:
    ax.set_xlabel("$x$")

for ax in axes[::4]:
    ax.set_ylabel("$u(x)$")

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="outside lower center",
    ncol=4,
    frameon=False,
)

fig.suptitle(
    "Debug A — BCs are part of the operator input",
    fontsize=13,
)

plt.show()

### Diagnosis A — interpret the results

1. **Compare errors.** Record the mean relative $L^2$ errors printed for the FNO **without** and **with** boundary-condition inputs. Which model performs better?
2. **Inspect the curves.** In the plotted examples, what happens near $x=0$ and $x=1$ when boundary conditions are omitted? Does providing them impose the endpoint values **exactly**?
3. **Use the counterexample.** For $f(x)=0$, can two different solutions arise from different $(b_0,b_1)$? Refer to the numerical sanity check above.
4. **Identify the cause.** Is the main limitation of the forcing-only model insufficient network capacity or missing input information? Explain in one sentence.


## Debug B — Recursive rollout (15 min)

Re-use the Burgers FNO from Part 1. It was trained on **one-step** pairs $u(0)\mapsto u(0.1)$, but is now applied recursively:

$$\widehat u_{n+1}=\mathcal G_\theta(\widehat u_n).$$

Inputs after the first step include the model's **own predictions** and later-time states. The reference rollout repeatedly solves the viscous Burgers PDE by Fourier pseudospectral discretization and RK4 for 21 steps of $\Delta t=0.1$. It is a numerical reference, not an exact solution.

### Prediction B (before rollout)

Do you expect errors to increase under repeated application of the learned one-step map? Give **one possible reason**.

In [ ]:
def burgers_reference_rollout(u0, n_steps):
    states = [np.asarray(u0, dtype=np.float32)]
    current = states[0]
    for _ in range(n_steps):
        current = solve_viscous_burgers(current, final_time=OPERATOR_DT)
        states.append(current)
    return np.stack(states, axis=0)

N_ROLLOUT = 21
rollout_initial = u0_64[N_TRAIN+N_VAL:]
reference_rollout = burgers_reference_rollout(rollout_initial, N_ROLLOUT)
state = torch.as_tensor(rollout_initial, dtype=torch.float32)
predicted_rollout = [state.clone()]

fno_burgers.eval()
with torch.no_grad():
    # TODO: recursively apply fno_burgers N_ROLLOUT times.
    # Reminder: network input shape is (batch, channel, space).
    ...

predicted_rollout = torch.stack(predicted_rollout, dim=0)
assert predicted_rollout.shape == reference_rollout.shape

In [ ]:
reference_rollout_t = torch.as_tensor(reference_rollout)
# First axis: time; second: sample; third: space.
diff_norm = torch.linalg.vector_norm(predicted_rollout-reference_rollout_t, dim=-1)
ref_norm = torch.linalg.vector_norm(reference_rollout_t, dim=-1)
initial_norm = ref_norm[0].clamp_min(1e-12)

rollout_relative = (diff_norm/ref_norm.clamp_min(1e-12)).mean(dim=1).numpy()
rollout_initial_relative = (diff_norm/initial_norm[None, :]).mean(dim=1).numpy()
rollout_absolute = diff_norm.mean(dim=1).numpy()
reference_amplitude = ref_norm.mean(dim=1).numpy()

rollout_times = np.arange(N_ROLLOUT+1)*OPERATOR_DT
fig, axes = plt.subplots(1, 2, figsize=(11.2, 3.8), constrained_layout=True)
axes[0].plot(rollout_times, rollout_relative, 'o-', label='relative to current state')
axes[0].plot(rollout_times, rollout_initial_relative, 's--', label='relative to initial state')
axes[0].set(xlabel='time', ylabel='mean normalized L2 error', title='Two denominators')
axes[0].legend(frameon=False)
axes[1].plot(rollout_times, rollout_absolute, 'o-', label='absolute rollout error')
axes[1].plot(rollout_times, reference_amplitude, '--', label='reference norm')
axes[1].set(xlabel='time', ylabel='mean discrete L2 norm', title='Error and solution decay')
axes[1].legend(frameon=False)
for ax in axes: ax.grid(alpha=0.2)
plt.show()

print('Final relative error (current norm):', f'{rollout_relative[-1]:.3%}')
print('Final error / initial norm:', f'{rollout_initial_relative[-1]:.3%}')

In [ ]:
sample = 3
steps_to_show = [
    0,
    7,
    15,
    20,
]

fig, axes = plt.subplots(
    1,
    4,
    figsize=(14, 3.5),
    sharey=True,
    constrained_layout=True,
)

for ax, step in zip(
    axes,
    steps_to_show,
):

    ax.plot(
        x64,
        reference_rollout[
            step,
            sample,
        ],
        label="reference",
    )

    ax.plot(
        x64,
        predicted_rollout[
            step,
            sample,
        ],
        "--",
        label="rollout",
    )

    ax.set_title(
        f"$t={step*OPERATOR_DT:.2f}$"
    )

    ax.set_xlabel("$x$")
    ax.grid(alpha=0.2)

axes[0].set_ylabel("$u(x)$")

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="outside upper center",
    ncol=2,
    frameon=False,
)

plt.show()

### Diagnosis B — interpret the results

1. **Read the rollout graph.** How does the prediction error change with the number of recursive steps? Is the increase monotonic?
2. **Compare normalizations.** Do the error curves normalized by $\|u(t)\|_2$ and by $\|u(0)\|_2$ evolve in the same way?
3. **Interpret the denominator.** How does the decreasing amplitude of a viscous Burgers solution affect the error normalized by $\|u(t)\|_2$? Use the absolute-error plot to support your answer.
4. **Explain the mechanism.** Why can a model trained on accurate one-step pairs perform worse when recursively applied to its own predictions?


# Final synthesis (5 min)

Complete the table before proposing a larger network:

| Observation | First diagnosis | Targeted test or remedy |
|---|---|---|
| Two different $u$ correspond to the same supplied $[f,x]$ | ? | ? |
| Small one-step error, poor long-time rollout | ? | ? |
| Similar accuracy at $N=32,64,128$ | What does this **not** establish? | What extra numerical study is needed? |

**Take-home:** The **representation** (PCA global coefficients versus FNO latent fields), the **available input information**, and the **evaluation protocol** are three separate design decisions. No architecture can learn a uniquely defined output from insufficient input data, and good one-step accuracy does not imply long-term rollout accuracy.

## Bonus — How to mitigate the identified limitations?

In Diagnoses A and B, we identified two different limitations of neural operators: incomplete input information and error accumulation during recursive predictions.

**Your task:** Propose strategies to mitigate these limitations. For each diagnosis, explain your approach and, if time permits, implement and evaluate one of your proposed solutions. You can take inspiration from the following ideas if needed: 



**Diagnosis A — Missing boundary conditions**

1. How could the boundary conditions be incorporated into the model?
2. Could we modify the loss function to encourage the predictions to satisfy the prescribed boundary conditions?

*Hint:* Consider adding the boundary values as input features and introducing a boundary-condition penalty in the training loss.



**Diagnosis B — Recursive rollout errors**

1. How could we modify the training procedure to improve long-term predictions?
2. Could an additional neural network be trained to predict and correct the errors accumulated during rollout?

*Hint:* Consider training on multiple consecutive time steps or learning a correction from the difference between predicted and reference trajectories.

**Optional implementation:** Choose one strategy, implement it, and compare its performance with the original model. Does your modification improve the results?